In [8]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go


import lime
import lime.lime_tabular

import shap

import librosa
import torch
from transformers import Wav2Vec2Model
import torch.nn as nn

In [3]:
import sys
sys.path.append("../w2v (Mohini)")
from wav2vec_model import Wav2Vec2Deepfake

In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [5]:
## ----Wave2Vec 2.0 Second Model----

# ESTABLISH MODEL 

# hyperparameters
SR = 16000
MAX_LEN = 2 * SR
BATCH_SIZE = 16
EPOCHS = 5
LR = 1e-5
NUM_WORKERS = 4

model = Wav2Vec2Deepfake().to(
        DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),
        lr=LR
    )
scaler = torch.amp.GradScaler()


# load model weight from checkpoint
model_path=r"C:\Users\eglha\CapitalOne_AudioDeepfake_Project\explainability (Hannah)\mohini_model.pth"
# load model path

checkpoint = torch.load(model_path, map_location=DEVICE)

if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'], strict=False)
else:
    model.load_state_dict(checkpoint, strict=False)

model.to(DEVICE)
model.eval()

print("Model loaded.")



Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.


C:\Users\eglha\AppData\Local\Temp\ipykernel_10500\165167494.py:23: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = torch.amp.GradScaler()


In [ ]:
def cache_audio_dataset(audio_dir):
    print(f"\nCaching: {audio_dir}")
    files = [
        f for f in (audio_dir)
        if f.endswith(".flac")
    ]
    for file in tqdm(files):
        cache_path = os.path.join(
            CACHE_DIR,
            file.replace(".flac", ".pt")
        )

        if os.path.exists(cache_path):
            continue

        audio_path = os.path.join(
            audio_dir,
            file
        )

        waveform = process_audio(
            audio_path
        )
        
        if waveform is None:
        
            continue
        
        torch.save(
            waveform,
            cache_path
        )
cache_audio_dataset(
        TRAIN_AUDIO_DIR
    )
train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True
    )

,true_label,prediction,score
0,0,0,0.000100
1,0,0,0.000558
2,0,0,0.000089
3,0,0,0.000115
4,0,0,0.000090
...,...,...,...
611824,0,0,0.000274
611825,0,0,0.000079
611826,0,0,0.000088
611827,0,0,0.000479


In [13]:
from pathlib import Path

audio_dir=Path(r"G:\My Drive\ASVSpoof_Data\unzipped2019\LA\LA\ASVspoof2019_LA_dev\flac")


In [16]:
def lime_prediction(audio_path):
    audio_path = str(audio_path) 

    model.eval()

    waveform, sr = librosa.load(
        audio_path,
        sr=SR,
        mono=True
    )

    waveform = torch.tensor(waveform, dtype=torch.float32)
    waveform = waveform.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(waveform)
        probs = torch.softmax(logits, dim=-1)[:, 1]

    return probs

for audio_path in audio_dir.glob("*.flac"):
    probs=lime_prediction(audio_path)
    print(audio_path.name, probs)

LA_D_9695291.flac tensor([0.4680])
LA_D_9695420.flac tensor([0.4581])
LA_D_9695805.flac tensor([0.4644])
LA_D_9696008.flac tensor([0.4564])
LA_D_9696470.flac tensor([0.4584])
LA_D_9697098.flac tensor([0.4685])
LA_D_9697274.flac tensor([0.4636])
LA_D_9697323.flac tensor([0.4592])
LA_D_9697774.flac tensor([0.4591])
LA_D_9698452.flac tensor([0.4677])
LA_D_9698557.flac tensor([0.4656])
LA_D_9698818.flac tensor([0.4553])
LA_D_9698822.flac tensor([0.4630])
LA_D_9699048.flac tensor([0.4585])
LA_D_9699074.flac tensor([0.4582])
LA_D_9699186.flac tensor([0.4719])
LA_D_9699380.flac tensor([0.4666])
LA_D_9699422.flac tensor([0.4628])
LA_D_9699768.flac tensor([0.4592])
LA_D_9699780.flac tensor([0.4669])
LA_D_9700502.flac tensor([0.4687])
LA_D_9700666.flac tensor([0.4637])
LA_D_9701091.flac tensor([0.4652])
LA_D_9701184.flac tensor([0.4646])
LA_D_9701451.flac tensor([0.4542])
LA_D_9701877.flac tensor([0.4675])
LA_D_9702949.flac tensor([0.4681])
LA_D_9703394.flac tensor([0.4639])
LA_D_9704152.flac te

KeyboardInterrupt: 